# Skincare at Sephora: From a Few Giants to a Crowded Field

**Question:** Which skincare brands received the most customer reviews on Sephora each year from 2018 to 2022, and did the leaders stay the same?

**Data:** Sephora skincare reviews (Kaggle), 2008 to March 2023. This analysis uses 2018 to 2022, the years with enough reviews and a complete calendar year.

**Definition:** "Top brand" means most reviews in that year. It does not mean most sales.

**Important:** The same review can appear under several product listings (minis, sets, sizes), so duplicates are removed before counting.

# Load and Remove Duplicates

In [11]:
import pandas as pd

# Folder holding the raw Kaggle files (not included in this repo)
DATA_DIR = "/Users/hazeezatadebayo/Downloads/sephora/"
FILES = ["reviews_0-250.csv", "reviews_250-500.csv", "reviews_500-750.csv",
         "reviews_750-1250.csv", "reviews_1250-end.csv"]
COLS = ["author_id", "brand_name", "rating", "submission_time", "review_text"]

parts = []
for f in FILES:
    d = pd.read_csv(DATA_DIR + f, usecols=COLS, low_memory=False)
    # Hash the text so duplicates can be found without keeping all text in memory
    d["text_hash"] = pd.util.hash_pandas_object(d["review_text"].fillna(""), index=False)
    parts.append(d.drop(columns="review_text"))
reviews = pd.concat(parts, ignore_index=True)

reviews["date"] = pd.to_datetime(reviews["submission_time"], errors="coerce")
reviews["year"] = reviews["date"].dt.year
print("Raw reviews:", f"{len(reviews):,}")
print("Date range:", reviews["date"].min().date(), "to", reviews["date"].max().date())

# A duplicate = same reviewer, brand, timestamp, rating and text
KEY = ["author_id", "brand_name", "submission_time", "rating", "text_hash"]
dup = reviews.duplicated(subset=KEY)
print(f"Duplicates removed: {dup.sum():,} ({dup.mean()*100:.1f}%)")

clean = reviews[~dup]
print(f"Unique reviews: {len(clean):,} | Brands: {clean['brand_name'].nunique()}")

# Analysis window: full calendar years 2018 to 2022
r = clean[clean["year"].between(2018, 2022)]
print(f"Reviews in the 2018 to 2022 window: {len(r):,}")

Raw reviews: 1,094,411
Date range: 2008-08-28 to 2023-03-21
Duplicates removed: 122,926 (11.2%)
Unique reviews: 971,485 | Brands: 142
Reviews in the 2018 to 2022 window: 745,281


# Brand rankings and concentration 

In [12]:
by = (r.groupby(["year", "brand_name"])["rating"]
        .agg(reviews="count", avg_rating="mean").reset_index())
by["avg_rating"] = by["avg_rating"].round(2)
by["rank"] = by.groupby("year")["reviews"].rank(ascending=False, method="min").astype(int)

total = by.groupby("year")["reviews"].sum()
def top_share(n):
    return by.groupby("year")["reviews"].apply(lambda s: s.nlargest(n).sum()) / total * 100

concentration = pd.DataFrame({
    "total_reviews": total,
    "top_1_share": top_share(1),
    "top_5_share": top_share(5),
    "top_10_share": top_share(10),
    "brands_1000plus": by[by["reviews"] >= 1000].groupby("year").size(),
    "brands_with_reviews": by.groupby("year").size(),
}).round(1).reset_index()
print(concentration.to_string(index=False))

 year  total_reviews  top_1_share  top_5_share  top_10_share  brands_1000plus  brands_with_reviews
 2018          87317          6.6         27.3          46.5               25                   72
 2019         129223          9.7         32.2          47.7               38                   86
 2020         182295          6.3         26.0          42.4               48                   99
 2021         174278          4.1         18.2          33.1               52                  116
 2022         172168          4.1         18.0          31.7               50                  134


# Checks behind each claim, plus the Tableau exports

In [13]:
def top(year, n):
    return list(by[(by["year"] == year) & (by["rank"] <= n)].sort_values("rank")["brand_name"])

top5_2019, top5_2022 = top(2019, 5), top(2022, 5)
print("2019 top 5:", top5_2019)
print("2022 top 5:", top5_2022)
print("Overlap between the two top-5 lists:", set(top5_2019) & set(top5_2022))
print("2019 top 10 still in the 2022 top 10:", set(top(2019, 10)) & set(top(2022, 10)))

ranks = by.pivot(index="brand_name", columns="year", values="rank")
revs = by.pivot(index="brand_name", columns="year", values="reviews")
for b in ["Drunk Elephant", "Tatcha", "Glow Recipe", "Shiseido", "Dermalogica"]:
    print(f"{b}: rank {int(ranks.loc[b, 2019])} -> {int(ranks.loc[b, 2022])}, "
          f"reviews {int(revs.loc[b, 2019]):,} -> {int(revs.loc[b, 2022]):,}")

# Exports used by the Tableau dashboard
top_brands = sorted(by[by["year"].isin([2019, 2022]) & (by["rank"] <= 10)]["brand_name"].unique())
grid = pd.MultiIndex.from_product([top_brands, range(2018, 2023)], names=["brand_name", "year"])
brands_long = by.set_index(["brand_name", "year"]).reindex(grid).reset_index()
brands_long["reviews"] = brands_long["reviews"].fillna(0).astype(int)
brands_long.to_csv(DATA_DIR + "skincare_brands_by_year.csv", index=False)
concentration.to_csv(DATA_DIR + "skincare_concentration.csv", index=False)
print("Exported 2 CSVs for Tableau.")

2019 top 5: ['Drunk Elephant', 'The Ordinary', 'Tatcha', 'LANEIGE', 'fresh']
2022 top 5: ['Glow Recipe', 'CLINIQUE', 'SEPHORA COLLECTION', 'Shiseido', 'Dermalogica']
Overlap between the two top-5 lists: set()
2019 top 10 still in the 2022 top 10: {'LANEIGE', 'The Ordinary', 'fresh'}
Drunk Elephant: rank 1 -> 19, reviews 12,558 -> 3,065
Tatcha: rank 3 -> 20, reviews 7,660 -> 2,993
Glow Recipe: rank 19 -> 1, reviews 2,257 -> 6,994
Shiseido: rank 34 -> 4, reviews 1,253 -> 5,680
Dermalogica: rank 27 -> 5, reviews 1,869 -> 5,116
Exported 2 CSVs for Tableau.


## Findings

- **2019 was the peak of concentration.** The five most-reviewed brands held 32.2% of skincare reviews, and Drunk Elephant alone held 9.7%.
- **The shift happened in 2020 and 2021.** The top-five share fell to 26.0%, then 18.2%, and held at 18.0% in 2022. The #1 brand in 2022 (Glow Recipe) held 4.1%.
- **The leaders changed.** None of the 2019 top five were in the 2022 top five. Drunk Elephant went from 1st to 19th and Tatcha from 3rd to 20th. Glow Recipe rose from 19th to 1st, Shiseido from 34th to 4th, and Dermalogica from 27th to 5th.
- **The field got bigger.** 50 brands had 1,000+ reviews in 2022 (38 in 2019), and 134 brands received a review (86 in 2019). Part of the drop in top-five share comes from more brands entering.

## Limitations

- Reviews are not sales.
- The data shows who moved, not why.
- The dataset only includes products listed when it was collected (through March 2023), so earlier years may be undercounted.
- Duplicates were identified by matching reviewer, brand, timestamp, rating and text, which is a reasonable method but not a perfect one.